# Phase 8 — Hyperparameter tuning

This notebook explains the controlled CatBoost and LightGBM search. The reusable implementation lives in `src/tune_models.py`; the notebook reads its auditable outputs so opening it does not silently rerun a multi-minute search.

**Hard guardrail:** the search uses only 10,670 fixed training rows. The 2,287 validation rows are used only after search for final-refit early stopping and comparison. The 2,287 test rows remain sealed.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

summary_path = PROJECT_ROOT / 'reports/metrics/hyperparameter_tuning_summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(summary['phase'])
print('Training / validation / test:', summary['split']['training_rows'], summary['split']['validation_rows'], summary['split']['test_rows'])
print('Test rows used:', summary['split']['test_rows_used'])

## 1. Why tune only two models?

Phase 7 found CatBoost strongest overall and LightGBM the most credible challenger. XGBoost showed unstable luxury-car errors, so spending the limited 8 GB compute budget on it was not justified. This is a deliberate narrowing based on validation diagnostics, not a hidden result.

## 2. Search contract

Each parameter set is evaluated with three-fold `StratifiedKFold`, using the fixed price-band label as the stratum. Models still train on `log1p(selling_price)`, but the ranking objective is mean absolute error after `expm1` returns predictions to original rupees.

Preprocessing is fold-safe: CatBoost medians and LightGBM imputation/one-hot encoding are refit inside each training fold.

In [ ]:
contract = summary['search_contract']
pd.Series(contract, name='value').to_frame()

## 3. Search spaces and sampled trials

The search is intentionally small: eight CatBoost trials and six LightGBM trials. Random seed 42 makes the sampled configurations reproducible.

In [ ]:
for model_name, space in summary['search_spaces'].items():
    print(f'\n{model_name}')
    display(pd.DataFrame({'parameter': space.keys(), 'candidate_values': [str(v) for v in space.values()]}))

trials = pd.read_csv(PROJECT_ROOT / 'reports/tables/phase8_tuning_trials.csv')
trials[['model', 'trial', 'rank', 'mean_cv_mae_inr', 'std_cv_mae_inr', 'training_seconds', 'parameters_json']].sort_values(['model', 'rank'])

In [ ]:
display(Image(filename=str(PROJECT_ROOT / 'reports/figures/22_tuning_cv_results.png')))

The error bars are one standard deviation across the three folds. Their overlap is a reminder that small score differences between configurations are uncertain; the search winner is a disciplined choice, not proof that nearby configurations are universally worse.

## 4. Best parameters

In [ ]:
best_rows = []
for model_name, result in summary['models'].items():
    best_rows.append({
        'model': model_name,
        'best_trial': result['best_trial'],
        'mean_cv_mae_inr': result['best_cv']['mean_mae_inr'],
        'std_cv_mae_inr': result['best_cv']['std_mae_inr'],
        'search_seconds': result['search_seconds'],
        'best_parameters': result['best_parameters'],
    })
pd.DataFrame(best_rows)

## 5. One fixed-validation comparison

After the search, each winning configuration is refit on all training rows. Early stopping watches the fixed validation split, and the resulting validation metrics are compared with the untouched Phase 7 results. This comparison does not feed back into another search.

In [ ]:
comparison = pd.read_csv(PROJECT_ROOT / 'reports/tables/phase8_tuned_validation_comparison.csv')
comparison[['model', 'mae_inr', 'rmse_inr', 'r2', 'median_absolute_error_inr', 'rmsle', 'within_20_percent_rate']]

In [ ]:
display(Image(filename=str(PROJECT_ROOT / 'reports/figures/23_tuned_validation_comparison.png')))
display(Image(filename=str(PROJECT_ROOT / 'reports/figures/24_tuned_validation_by_price_band.png')))

## 6. Conclusion

Tuned CatBoost lowers validation MAE from **₹89,656 to ₹88,384**, a **₹1,272 (1.42%)** improvement. RMSE and RMSLE also improve slightly.

Tuned LightGBM raises validation MAE by ₹1,038 (1.07%) and has a much worse RMSE, so it is retained for auditability but not promoted.

Tuned CatBoost is therefore the Phase 9 explanation-and-recommendation candidate. It is not yet called the final model, and no test metric has been calculated.

## Reproducing the full search

From the project root, intentionally rerun the complete search with:

```bash
python -m src.tune_models
```

This takes a few minutes and overwrites the Phase 8 result artifacts with a deterministic rerun.